In [ ]:
# ==== 0. Cài đặt thư viện (chạy 1 lần / session) ====
!git clone -q https://github.com/soCzech/TransNetV2.git
!pip install -q ffmpeg-python pillow opencv-python-headless
!pip install -q open_clip_torch
!pip install -q google-auth google-auth-oauthlib google-api-python-client
!pip install -q boto3
# Kaggle image thường đã có sẵn tensorflow + torch với CUDA khớp driver GPU T4 -> KHÔNG
# cần tự cài lại torch/tensorflow (khác với máy local RTX 3060 phải tự chọn bản CUDA).
print("Xong bước cài đặt.")


In [ ]:
# ==== 1. Imports ====
import os
import sys
import io
import json
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload, MediaIoBaseUpload

sys.path.append("/kaggle/working/TransNetV2/inference")
from transnetv2 import TransNetV2
import open_clip


In [ ]:
# ==== 2. Cấu hình đường dẫn ====
# chứa các file drive_credentials.json, drive_token.json, drive_folders.json, aws_credentials.json
INPUT_DATASET_DIR = Path("/kaggle/input/pov-drive-credentials")

WORKING_DIR = Path("/kaggle/working")
RAW_VIDEOS_LOCAL = WORKING_DIR / "raw_videos"
KEYFRAMES_LOCAL = WORKING_DIR / "keyframes"
STATE_LOCAL = WORKING_DIR / "state"  # bản ghi cục bộ, KHÔNG phải nguồn resume chính (state giờ ở S3)

for p in (RAW_VIDEOS_LOCAL, KEYFRAMES_LOCAL, STATE_LOCAL):
    p.mkdir(parents=True, exist_ok=True)

CREDENTIALS_SRC = INPUT_DATASET_DIR / "drive_credentials.json"
TOKEN_SRC = INPUT_DATASET_DIR / "drive_token.json"  # JSON thuần, không dùng pickle (tránh lỗi version google-auth)
FOLDERS_JSON_SRC = INPUT_DATASET_DIR / "drive_folders.json"

for p in (CREDENTIALS_SRC, TOKEN_SRC, FOLDERS_JSON_SRC):
    assert p.exists(), f"Không tìm thấy {p} — kiểm tra lại INPUT_DATASET_DIR và tên dataset đã Add Data."

# ==== Cấu hình S3 (đích upload keyframes) ====
# Bucket + region đúng như đã tạo bằng Terraform (terraform/main.tf, provider.tf).
S3_BUCKET = "agent-force-video-bucket"
S3_REGION = "ap-southeast-1"
S3_PREFIX = ""  # prefix gốc trong bucket; để "" nếu ghi thẳng vào root

# AWS credentials: đặt trong CÙNG Kaggle Dataset như drive_*.json.
# Nội dung file aws_credentials.json (lấy giá trị từ `terraform output`):
#   {
#     "aws_access_key_id": "AKIA...",           # = terraform output video_processing_access_key_id
#     "aws_secret_access_key": "...",           # = terraform output -raw video_processing_secret_access_key
#     "region": "ap-southeast-1"
#   }
AWS_CREDENTIALS_SRC = INPUT_DATASET_DIR / "aws_credentials.json"

print("Đường dẫn OK.")


In [ ]:
# ==== 3. Google Drive helpers (viết thẳng ở đây, KHÔNG import file .py rời) ====
SCOPES = ["https://www.googleapis.com/auth/drive"]


def get_drive_service_kaggle(credentials_src: Path, token_src: Path, token_write_dir: Path):
    """Load credentials + token ĐÃ TẠO SẴN từ máy local (token ở dạng JSON thuần,
    KHÔNG dùng pickle -- pickle của google-auth Credentials gắn chặt với version thư
    viện lúc tạo ra nó, dễ vỡ khi unpickle ở môi trường khác version, như Kaggle).
    Trên Kaggle (headless) sẽ KHÔNG bao giờ mở luồng OAuth xin quyền qua trình duyệt
    (flow.run_local_server) -- nếu token không load/refresh được thì báo lỗi rõ ràng
    thay vì treo notebook."""
    token_write_dir.mkdir(parents=True, exist_ok=True)

    data = json.loads(token_src.read_text())
    creds = Credentials(
        token=data.get("token"),
        refresh_token=data.get("refresh_token"),
        token_uri=data.get("token_uri"),
        client_id=data.get("client_id"),
        client_secret=data.get("client_secret"),
        scopes=data.get("scopes"),
    )
    if not creds.valid:
        if creds.refresh_token:
            creds.refresh(Request())
        else:
            raise RuntimeError(
                "Token Drive không có refresh_token để tự làm mới. "
                "Chạy lại OAuth flow TRÊN MÁY LOCAL (nơi có trình duyệt) để tạo "
                "drive_token.pickle mới, convert lại sang JSON (convert_token_to_json.py), "
                "rồi upload lại vào Kaggle Dataset."
            )
    return build("drive", "v3", credentials=creds)


def get_or_create_folder(service, name: str, parent_id: str, cache_path: Path) -> str:
    """Giống hệt logic gốc, chỉ khác: cache_path truyền vào tường minh (không dựa vào
    __file__ như bản gốc, vì trong notebook không có __file__ ổn định)."""
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    cache_key = f"{parent_id}/{name}"
    if cache_key in cache:
        return cache[cache_key]

    query = (
        f"name = '{name}' and '{parent_id}' in parents "
        "and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    )
    response = service.files().list(q=query, fields="files(id, name)").execute()
    files = response.get("files", [])
    if files:
        folder_id = files[0]["id"]
    else:
        metadata = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
        folder = service.files().create(body=metadata, fields="id").execute()
        folder_id = folder["id"]

    cache[cache_key] = folder_id
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(cache, indent=2))
    return folder_id


def upload_file_to_drive(service, local_path: Path, folder_id: str, name: str = None) -> str:
    file_metadata = {"name": name or local_path.name, "parents": [folder_id]}
    media = MediaFileUpload(str(local_path), resumable=True, chunksize=50 * 1024 * 1024)
    request = service.files().create(body=file_metadata, media_body=media, fields="id")
    response = None
    while response is None:
        status, response = request.next_chunk()
    return response["id"]


def upload_json_to_drive(service, data, filename: str, folder_id: str, overwrite_existing_id: str = None) -> str:
    payload = json.dumps(data, indent=2, ensure_ascii=False).encode("utf-8")
    media = MediaIoBaseUpload(io.BytesIO(payload), mimetype="application/json", resumable=False)
    if overwrite_existing_id:
        service.files().update(fileId=overwrite_existing_id, media_body=media).execute()
        return overwrite_existing_id
    file_metadata = {"name": filename, "parents": [folder_id]}
    response = service.files().create(body=file_metadata, media_body=media, fields="id").execute()
    return response["id"]


def download_file_from_drive(service, file_id: str, dest_path: Path):
    request = service.files().get_media(fileId=file_id)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dest_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest_path


def download_json_from_drive(service, file_id: str):
    request = service.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return json.loads(buf.getvalue().decode("utf-8"))


def list_files_in_folder(service, folder_id: str, name_contains: str = None):
    files = []
    page_token = None
    q = f"'{folder_id}' in parents and trashed = false"
    if name_contains:
        q += f" and name contains '{name_contains}'"
    while True:
        response = service.files().list(
            q=q, spaces="drive", fields="nextPageToken, files(id, name)", pageToken=page_token,
        ).execute()
        files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return files


def find_file_by_name(service, name: str, folder_id: str):
    query = f"name = '{name}' and '{folder_id}' in parents and trashed = false"
    response = service.files().list(q=query, fields="files(id, name)").execute()
    files = response.get("files", [])
    return files[0]["id"] if files else None


In [ ]:
# ==== 3b. S3 helpers (boto3) ====
# Keyframes, metadata.json và state resume giờ lưu trên S3 thay vì Drive.
# (Video nguồn raw_videos/ vẫn đọc từ Drive như cũ.)
import boto3
from botocore.config import Config as BotoConfig
from botocore.exceptions import ClientError


def get_s3_client():
    """Đọc AWS credentials từ file trong Kaggle Dataset (aws_credentials.json).
    Fallback: Kaggle Secrets, rồi tới credentials mặc định của môi trường (env/~/.aws)."""
    access_key = secret_key = region = None

    if AWS_CREDENTIALS_SRC.exists():
        data = json.loads(AWS_CREDENTIALS_SRC.read_text())
        access_key = data.get("aws_access_key_id")
        secret_key = data.get("aws_secret_access_key")
        region = data.get("region")
    else:
        try:
            from kaggle_secrets import UserSecretsClient
            us = UserSecretsClient()
            access_key = us.get_secret("AWS_ACCESS_KEY_ID")
            secret_key = us.get_secret("AWS_SECRET_ACCESS_KEY")
        except Exception:
            pass

    region = region or S3_REGION
    cfg = BotoConfig(region_name=region, retries={"max_attempts": 5, "mode": "standard"})
    if access_key and secret_key:
        return boto3.client(
            "s3", aws_access_key_id=access_key, aws_secret_access_key=secret_key, config=cfg,
        )
    # Không có key tường minh -> dùng credentials mặc định của môi trường.
    return boto3.client("s3", config=cfg)


def s3_key(*parts) -> str:
    """Ghép key S3, tự bỏ dấu / thừa và tôn trọng S3_PREFIX."""
    segs = []
    if S3_PREFIX.strip("/"):
        segs.append(S3_PREFIX.strip("/"))
    segs += [str(p).strip("/") for p in parts if str(p).strip("/")]
    return "/".join(segs)


def upload_file_to_s3(s3, local_path: Path, key: str, content_type: str = None) -> str:
    extra = {"ContentType": content_type} if content_type else None
    s3.upload_file(str(local_path), S3_BUCKET, key, ExtraArgs=extra)
    return key


def upload_json_to_s3(s3, data, key: str) -> str:
    payload = json.dumps(data, indent=2, ensure_ascii=False).encode("utf-8")
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=payload, ContentType="application/json")
    return key


def download_json_from_s3(s3, key: str):
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return json.loads(obj["Body"].read().decode("utf-8"))


def s3_object_exists(s3, key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise


In [ ]:
# ==== 4. Kết nối Drive (đọc raw_videos) + khởi tạo S3 client (đích upload) ====
service = get_drive_service_kaggle(CREDENTIALS_SRC, TOKEN_SRC, STATE_LOCAL)
print("Đã kết nối Google Drive.")

drive_folders = json.loads(FOLDERS_JSON_SRC.read_text())
ROOT_FOLDER_ID = None
RAW_VIDEOS_FOLDER_ID = None
for key, folder_id in drive_folders.items():
    root_id, name = key.split("/")
    ROOT_FOLDER_ID = root_id
    if name == "raw_videos":
        RAW_VIDEOS_FOLDER_ID = folder_id

assert RAW_VIDEOS_FOLDER_ID, "Thiếu raw_videos trong drive_folders.json"
print("Root folder id      :", ROOT_FOLDER_ID)
print("raw_videos folder id:", RAW_VIDEOS_FOLDER_ID)

# Kết nối S3 và kiểm tra quyền truy cập bucket ngay từ đầu.
s3 = get_s3_client()
s3.head_bucket(Bucket=S3_BUCKET)
print("Đã kết nối S3 bucket:", S3_BUCKET, "(region", S3_REGION + ")")


In [ ]:
# ==== 5. Cấu hình tham số pipeline (y hệt notebook gốc) ====
CONFIG = {
    "transition_threshold": 0.75,
    "sampling_step": 15,
    "blur_threshold": 100,
    "duplicate_threshold": 0.7,
    "chunk_similarity_threshold": 0.83,
    "keyframes_per_chunk_divisor": 6,
    "clip_model_name": "ViT-B-32",
    "clip_pretrained": "laion2b_s34b_b79k",
    "delete_local_video_after_processing": True,
    "delete_local_keyframes_after_upload": True,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[CANH BAO] Không thấy GPU -- kiểm tra lại Settings > Accelerator = GPU T4 rồi Restart session.")


In [ ]:
# ==== 6. Load model (1 lần, dùng lại cho toàn bộ video) ====
print("Loading TransNetV2 ...")
transnet_model = TransNetV2()

print("Loading CLIP ...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CONFIG["clip_model_name"], pretrained=CONFIG["clip_pretrained"],
)
clip_model = clip_model.to(device)
clip_model.eval()
print("Models loaded.")


In [ ]:
# ==== 7. Các hàm xử lý pipeline (logic y hệt keyframe-extraction.ipynb gốc) ====

def extract_video_metadata(video_path: str):
    if not os.path.exists(video_path):
        print(f"[PREPROCESSING] The video is not found in the given path {video_path}")
        return None
    vid_capture = cv2.VideoCapture(video_path)
    if vid_capture.isOpened() is False:
        print("[PREPROCESSING] Error in opening the video file")
        return None
    fps = vid_capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(vid_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(vid_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(vid_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps else 0
    vid_capture.release()
    return {"fps": fps, "frame_count": frame_count, "width": width, "height": height, "duration": duration}


def extract_transition_frame(video_path: str, threshold: float, model):
    video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video_path)
    transition_frames = np.where(all_frame_predictions > threshold)[0]
    if len(transition_frames) == 0:
        return []
    transition_segment = []
    start_flag = transition_frames[0]
    end_flag = transition_frames[0]
    for frame in transition_frames[1:]:
        if end_flag + 1 == frame:
            end_flag = frame
        else:
            transition_segment.append((int(start_flag), int(end_flag)))
            start_flag = frame
            end_flag = frame
    transition_segment.append((int(start_flag), int(end_flag)))
    return transition_segment


def extract_scene(frame_count: int, transition_segment):
    scene_segment = []
    flag = 0
    for start, end in transition_segment:
        if flag <= (start - 1):
            scene_segment.append((flag, start - 1))
        flag = end + 1
    if flag < frame_count:
        scene_segment.append((flag, frame_count - 1))
    return scene_segment


def blur_score(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def histogram_similarity(frame1, frame2):
    hsv1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2HSV)
    hsv2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2HSV)
    hist1 = cv2.calcHist([hsv1], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    hist2 = cv2.calcHist([hsv2], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist1, hist1, norm_type=cv2.NORM_L1)
    cv2.normalize(hist2, hist2, norm_type=cv2.NORM_L1)
    return cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)


def extract_candidate_keyframes(video_path, scene_segment, sampling_step=15, blur_threshold=100, duplicate_threshold=0.7):
    cap = cv2.VideoCapture(video_path)
    scene_metadata = []
    for scene_id, (start, end) in enumerate(scene_segment):
        if start >= end:
            continue
        selected_frames = []
        keyframes = []
        for frame_idx in range(start, end + 1, sampling_step):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                continue
            if blur_score(frame) < blur_threshold:
                continue
            if len(selected_frames) == 0:
                selected_frames.append(frame)
                keyframes.append(frame_idx)
                continue
            duplicate = False
            for previous_frame in selected_frames:
                if histogram_similarity(previous_frame, frame) >= duplicate_threshold:
                    duplicate = True
                    break
            if duplicate:
                continue
            selected_frames.append(frame)
            keyframes.append(frame_idx)
        scene_metadata.append({
            "scene_id": scene_id, "start_frame": start, "end_frame": end,
            "total_candidate": len(keyframes), "candidate_keyframes": keyframes,
        })
    cap.release()
    return scene_metadata


def extract_candidate_embeddings(video_path, candidate_keyframes, clip_model_tuple):
    model, preprocess = clip_model_tuple
    device_ = next(model.parameters()).device
    model.eval()
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    scene_embeddings = []
    with torch.no_grad():
        for scene in candidate_keyframes:
            frame_embeddings = []
            for frame_idx in scene["candidate_keyframes"]:
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ret, frame = cap.read()
                if not ret:
                    print(f"[Warning] Cannot read frame {frame_idx}")
                    continue
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image = Image.fromarray(rgb)
                image = preprocess(image).unsqueeze(0).to(device_)
                embedding = model.encode_image(image)
                embedding = embedding / embedding.norm(dim=-1, keepdim=True)
                embedding = embedding.squeeze(0).detach().cpu().numpy().astype(np.float32)
                frame_embeddings.append({"frame_idx": frame_idx, "embedding": embedding})
            scene_embeddings.append({
                "scene_id": scene["scene_id"], "start_frame": scene["start_frame"],
                "end_frame": scene["end_frame"], "frames": frame_embeddings,
            })
    cap.release()
    return scene_embeddings


def temporal_semantic_chunking(candidate_embeddings, similarity_threshold=0.83):
    scene_chunks = []
    for scene in candidate_embeddings:
        frames = scene["frames"]
        if len(frames) == 0:
            continue
        chunks = []
        current_chunk = [frames[0]]
        for i in range(1, len(frames)):
            similarity = cosine_similarity(
                frames[i - 1]["embedding"].reshape(1, -1), frames[i]["embedding"].reshape(1, -1)
            )[0][0]
            if similarity >= similarity_threshold:
                current_chunk.append(frames[i])
            else:
                chunks.append(current_chunk)
                current_chunk = [frames[i]]
        chunks.append(current_chunk)
        scene_chunks.append({
            "scene_id": scene["scene_id"], "start_frame": scene["start_frame"],
            "end_frame": scene["end_frame"], "chunks": chunks,
        })
    return scene_chunks


def extract_keyframes(scene_chunks, divisor=6):
    scene_keyframes = []
    for scene in scene_chunks:
        final_keyframes = []
        for chunk in scene["chunks"]:
            n = len(chunk)
            if n == 0:
                continue
            k = int(np.ceil(n / divisor))
            segments = np.array_split(chunk, k)
            for segment in segments:
                if len(segment) == 0:
                    continue
                embeddings = np.stack([frame["embedding"] for frame in segment])
                centroid = embeddings.mean(axis=0)
                distances = np.linalg.norm(embeddings - centroid, axis=1)
                best = np.argmin(distances)
                final_keyframes.append(segment[best]["frame_idx"])
        scene_keyframes.append({
            "scene_id": scene["scene_id"], "start_frame": scene["start_frame"],
            "end_frame": scene["end_frame"], "num_keyframes": len(final_keyframes),
            "keyframes": sorted(final_keyframes),
        })
    return scene_keyframes


In [ ]:
# ==== 8. Lưu keyframe local + build metadata.json + upload lên S3 ====
# Layout trên S3:
#   keyframes/<video_id>/<frame_idx>.jpg
#   keyframes/<video_id>/metadata.json   (mỗi phần tử CHỈ gồm frame_idx + video_id)

def save_keyframes_local(video_path, scene_keyframes, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Cannot open video.")
    saved_files = []
    for scene in scene_keyframes:
        for frame_idx in scene["keyframes"]:
            frame_idx = int(frame_idx)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()
            if not success:
                print(f"[Warning] Cannot read frame {frame_idx}")
                continue
            filename = output_dir / f"{frame_idx:06d}.jpg"
            if not cv2.imwrite(str(filename), frame):
                print("Cannot save", filename)
                continue
            saved_files.append((frame_idx, filename))
    cap.release()
    return saved_files


def upload_keyframes_to_s3(video_id: str, saved_files, s3):
    metadata = []
    for frame_idx, filepath in sorted(saved_files, key=lambda x: x[0]):
        key = s3_key("keyframes", video_id, filepath.name)
        upload_file_to_s3(s3, filepath, key, content_type="image/jpeg")
        metadata.append({"frame_idx": frame_idx, "video_id": video_id})
    meta_key = s3_key("keyframes", video_id, "metadata.json")
    upload_json_to_s3(s3, metadata, meta_key)
    keyframes_prefix = s3_key("keyframes", video_id)
    return keyframes_prefix, metadata


In [ ]:
# ==== 9. State resumable — lưu trên S3 ====
# Kaggle notebook có thể bị ngắt/hết giờ session bất cứ lúc nào -> /kaggle/working
# không đảm bảo còn nguyên. Nguồn resume thật lấy từ S3.
PROCESSED_STATE_KEY = s3_key("state", "processed_videos.json")


def load_processed_state():
    if s3_object_exists(s3, PROCESSED_STATE_KEY):
        print(f"Đang tải state resume từ s3://{S3_BUCKET}/{PROCESSED_STATE_KEY} ...")
        return download_json_from_s3(s3, PROCESSED_STATE_KEY)
    return {}


def save_processed_state(state):
    upload_json_to_s3(s3, state, PROCESSED_STATE_KEY)


In [ ]:
# ==== 10. Hàm xử lý toàn bộ 1 video (download từ Drive -> extract -> save -> upload S3) ====

def process_video(video_id: str, drive_file_id: str, service):
    t0 = time.time()
    local_video_path = RAW_VIDEOS_LOCAL / f"{video_id}.mp4"
    local_keyframes_dir = KEYFRAMES_LOCAL / video_id

    try:
        print(f"\n=== [{video_id}] Downloading video từ Drive ===")
        download_file_from_drive(service, drive_file_id, local_video_path)

        print(f"[{video_id}] Đọc metadata video ...")
        metadata = extract_video_metadata(str(local_video_path))
        if metadata is None:
            print(f"[{video_id}] Bỏ qua: không đọc được video.")
            return None
        frame_count = metadata["frame_count"]

        print(f"[{video_id}] Phát hiện chuyển cảnh (TransNetV2) ...")
        transition_segment = extract_transition_frame(str(local_video_path), CONFIG["transition_threshold"], transnet_model)
        scene_segment = extract_scene(frame_count, transition_segment)
        print(f"[{video_id}] Có {len(scene_segment)} scene.")

        print(f"[{video_id}] Trích candidate keyframes ...")
        candidate_keyframes = extract_candidate_keyframes(
            str(local_video_path), scene_segment,
            sampling_step=CONFIG["sampling_step"], blur_threshold=CONFIG["blur_threshold"],
            duplicate_threshold=CONFIG["duplicate_threshold"],
        )

        print(f"[{video_id}] Tính CLIP embedding ...")
        candidate_embeddings = extract_candidate_embeddings(
            str(local_video_path), candidate_keyframes, (clip_model, clip_preprocess)
        )

        print(f"[{video_id}] Temporal semantic chunking ...")
        scene_chunks = temporal_semantic_chunking(candidate_embeddings, similarity_threshold=CONFIG["chunk_similarity_threshold"])

        print(f"[{video_id}] Chọn keyframe đại diện ...")
        scene_keyframes = extract_keyframes(scene_chunks, divisor=CONFIG["keyframes_per_chunk_divisor"])

        total_keyframes = sum(s["num_keyframes"] for s in scene_keyframes)
        print(f"[{video_id}] Tổng cộng {total_keyframes} keyframes.")

        print(f"[{video_id}] Lưu keyframes local ...")
        saved_files = save_keyframes_local(str(local_video_path), scene_keyframes, local_keyframes_dir)

        print(f"[{video_id}] Upload keyframes + metadata.json lên S3 ...")
        keyframes_prefix, meta = upload_keyframes_to_s3(video_id, saved_files, s3)

        print(f"[{video_id}] Xong trong {time.time() - t0:.1f}s -> s3://{S3_BUCKET}/{keyframes_prefix}/")
        return keyframes_prefix, len(meta)

    finally:
        if CONFIG["delete_local_video_after_processing"] and local_video_path.exists():
            local_video_path.unlink()
        if CONFIG["delete_local_keyframes_after_upload"] and local_keyframes_dir.exists():
            shutil.rmtree(local_keyframes_dir, ignore_errors=True)


In [ ]:
# ==== 11. Quét raw_videos/ trên Drive và chạy hàng loạt ====
# LIMIT: giới hạn số video xử lý trong 1 lần chạy session này (None = chạy hết).
# Đặt 1 số nhỏ (vd 5) để test trước khi thả full, vì Kaggle T4 free có hạn ~30h/tuần.
LIMIT = None

raw_video_files = list_files_in_folder(service, RAW_VIDEOS_FOLDER_ID)
raw_video_files.sort(key=lambda f: f["name"])
print(f"Tìm thấy {len(raw_video_files)} video trong raw_videos/ trên Drive.")

processed_state = load_processed_state()
print(f"Đã xử lý trước đó: {len(processed_state)} video.")

count_this_run = 0
for f in raw_video_files:
    if LIMIT is not None and count_this_run >= LIMIT:
        print(f"Đã chạm LIMIT={LIMIT} cho session này, dừng lại (chạy lại notebook sẽ resume tiếp).")
        break

    video_id = Path(f["name"]).stem

    if video_id in processed_state:
        print(f"[{video_id}] Đã xử lý trước đó, bỏ qua.")
        continue

    result = process_video(video_id, f["id"], service)
    count_this_run += 1

    if result is not None:
        keyframes_prefix, num_keyframes = result
        processed_state[video_id] = {
            "drive_file_id": f["id"],
            "s3_bucket": S3_BUCKET,
            "keyframes_prefix": keyframes_prefix,
            "num_keyframes": num_keyframes,
        }
        save_processed_state(processed_state)  # ghi ngay -> resume được dù bị ngắt giữa chừng

print(f"\nHoàn tất session này. Tổng đã xử lý: {len(processed_state)}/{len(raw_video_files)} video.")


## 12. (Tuỳ chọn) Test nhanh với 1 video trước khi thả full

Nếu muốn thử pipeline trên đúng 1 video (không đụng tới state resume/vòng lặp ở trên),
uncomment và chạy cell dưới.

In [ ]:
# test_file = raw_video_files[0]
# test_video_id = Path(test_file["name"]).stem
# process_video(test_video_id, test_file["id"], service)
